# Repository note
This is a code-only copy of an exploratory notebook. Saved outputs were removed for public sharing. The required CSV datasets are not included; update the local/Colab paths in the cells before running. The original execution environment and full dataset provenance were not supplied, so a fresh run is not guaranteed to reproduce prior results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# **PreProcessing**

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Network_steganography_Project/FINAL_MASTER_RAW_DATASET.csv")

In [ ]:
# =====================================================
# COMPLETE PREPROCESSING PIPELINE
# Transformer-Based Network Steganography Detector
# =====================================================

import pandas as pd
import numpy as np
import math
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# =====================================================
# 1. LOAD DATASET
# =====================================================

df = pd.read_csv("/content/drive/MyDrive/Network_steganography_Project/FINAL_MASTER_RAW_DATASET.csv", low_memory=False)

print("Original Shape:", df.shape)

# =====================================================
# 2. REMOVE UNNECESSARY COLUMNS
# =====================================================

cols_to_drop = ["dataset_name"]

for col in cols_to_drop:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

print("Removed dataset_name column (if existed)")

# =====================================================
# 3. HANDLE DUPLICATE DOMAIN LENGTH FEATURE
# =====================================================

if "dns_domain_name_length" in df.columns and "dns_domain_name_len" in df.columns:

    print("Both domain length columns exist. Checking if identical...")

    if (df["dns_domain_name_length"] == df["dns_domain_name_len"]).all():
        print("Columns are identical. Removing dns_domain_name_length")
        df.drop(columns=["dns_domain_name_length"], inplace=True)
    else:
        print("Columns are different. Keeping both.")

# =====================================================
# 4. DNS LEXICAL FEATURE ENGINEERING
# =====================================================

def shannon_entropy(text):

    if len(text) == 0:
        return 0

    freq = Counter(text)
    probs = [v / len(text) for v in freq.values()]
    return -sum(p * math.log2(p) for p in probs)


def extract_domain_features(domain):

    domain = str(domain)

    length = len(domain)
    digits = sum(c.isdigit() for c in domain)
    letters = sum(c.isalpha() for c in domain)
    special = sum(not c.isalnum() for c in domain)

    unique_chars = len(set(domain))
    entropy = shannon_entropy(domain)

    return pd.Series([
        length,
        digits,
        letters,
        special,
        unique_chars,
        entropy,
        digits/length if length > 0 else 0,
        special/length if length > 0 else 0
    ])


dns_domain_cols = [
    "dns_domain_name",
    "uni_gram_domain_name",
    "bi_gram_domain_name",
    "tri_gram_domain_name"
]


for col in dns_domain_cols:

    if col in df.columns:

        df[col] = df[col].fillna("").astype(str)

        df[[f"{col}_len",
            f"{col}_digit_count",
            f"{col}_letter_count",
            f"{col}_special_count",
            f"{col}_unique_chars",
            f"{col}_entropy",
            f"{col}_digit_ratio",
            f"{col}_special_ratio"]] = df[col].apply(extract_domain_features)


# Drop heavy text columns
df.drop(columns=[c for c in dns_domain_cols if c in df.columns], inplace=True)

print("After DNS Feature Engineering:", df.shape)

# =====================================================
# 5. SIMPLE ENCODING FOR LOW CARDINALITY DNS FIELDS
# =====================================================

categorical_cols = [
    "dns_top_level_domain",
    "dns_second_level_domain",
    "ans_resource_record_type",
    "ans_resource_record_class"
]

for col in categorical_cols:

    if col in df.columns:

        df[col] = df[col].fillna("unknown").astype(str)

        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

print("Categorical DNS fields encoded")

# =====================================================
# 6. ENCODE PROTOCOL LABEL
# =====================================================

protocol_encoder = LabelEncoder()

df["label_protocol_enc"] = protocol_encoder.fit_transform(df["label_protocol"])

df.drop(columns=["label_protocol"], inplace=True)

print("Protocol encoded")

# =====================================================
# 7. BALANCE DATA PER PROTOCOL + BINARY
# =====================================================

balanced_list = []

for protocol in df["label_protocol_enc"].unique():

    subset = df[df["label_protocol_enc"] == protocol]

    normal = subset[subset["label_binary"] == 0]
    covert = subset[subset["label_binary"] == 1]

    if len(normal) < len(covert):

        normal = resample(
            normal,
            replace=True,
            n_samples=len(covert),
            random_state=42
        )

    else:

        normal = normal.sample(len(covert), random_state=42)

    balanced_protocol = pd.concat([normal, covert])
    balanced_list.append(balanced_protocol)


df_balanced = pd.concat(balanced_list)

print("After Protocol + Binary Balancing:")
print(df_balanced["label_binary"].value_counts())

# =====================================================
# 8. BALANCE PER TECHNIQUE
# =====================================================

min_tech = df_balanced["label_technique"].value_counts().min()

tech_list = []

for tech, group in df_balanced.groupby("label_technique"):

    tech_list.append(group.sample(min_tech, random_state=42))


df_balanced = pd.concat(tech_list)

print("After Technique Balance:")
print(df_balanced["label_technique"].value_counts())

# =====================================================
# 9. CLEAN DATA AND FORCE NUMERIC
# =====================================================

df_balanced.replace([np.inf, -np.inf], np.nan, inplace=True)

df_balanced.fillna(0, inplace=True)

label_cols = [
    "label_binary",
    "label_protocol_enc",
    "label_technique"
]

for col in df_balanced.columns:

    if col not in label_cols:

        df_balanced[col] = pd.to_numeric(df_balanced[col], errors='coerce')


df_balanced.fillna(0, inplace=True)

print("Data cleaned")

# =====================================================
# 10. FEATURE SCALING
# =====================================================

feature_cols = [c for c in df_balanced.columns if c not in label_cols]

scaler = StandardScaler()

df_balanced[feature_cols] = scaler.fit_transform(df_balanced[feature_cols])

print("Feature scaling completed")

# =====================================================
# 11. SAVE FINAL PREPROCESSED DATASET
# =====================================================

df_balanced.to_csv("preprocessed_master_dataset.csv", index=False)

print("Saved: preprocessed_master_dataset.csv")
print("Final Shape:", df_balanced.shape)

# =====================================================
# 12. TRAIN TEST SPLIT (STRATIFIED BY TECHNIQUE)
# =====================================================

X = df_balanced[feature_cols]

y_binary = df_balanced["label_binary"]
y_protocol = df_balanced["label_protocol_enc"]
y_technique = df_balanced["label_technique"]


X_train, X_test, yb_train, yb_test, yp_train, yp_test, yt_train, yt_test = train_test_split(
    X,
    y_binary,
    y_protocol,
    y_technique,
    test_size=0.2,
    random_state=42,
    stratify=y_technique
)


train_df = X_train.copy()
train_df["label_binary"] = yb_train.values
train_df["label_protocol_enc"] = yp_train.values
train_df["label_technique"] = yt_train.values


test_df = X_test.copy()
test_df["label_binary"] = yb_test.values
test_df["label_protocol_enc"] = yp_test.values
test_df["label_technique"] = yt_test.values


train_df.to_csv("train_dataset.csv", index=False)
test_df.to_csv("test_dataset.csv", index=False)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

print("Preprocessing Completed Successfully 🚀")

In [ ]:
import os

print(os.listdir())

In [ ]:
import pandas as pd

# Load dataset
new= pd.read_csv("/content/preprocessed_master_dataset.csv")

# 1️⃣ Shape (rows, columns)
print("Dataset Shape (rows, columns):", new.shape)

# 2️⃣ Number of rows
print("Total Rows:", new.shape[0])

# 3️⃣ Total Columns
print("Total Columns:", new.shape[1])

# 4️⃣ Feature Columns Only (excluding labels)
label_cols = ["label_binary", "label_protocol_enc", "label_technique"]
feature_cols = [col for col in new.columns if col not in label_cols]

print("Total Feature Columns:", len(feature_cols))

In [ ]:
new.columns.to_list()

In [ ]:
print(new["character_distribution"].dtype)

In [ ]:
non_numeric = new.select_dtypes(exclude=['number'])
print(non_numeric.columns)

In [ ]:
print("Final Shape:", new.shape)

# Check remaining non-numeric columns
non_numeric = new.select_dtypes(exclude=['number'])
print("Non-numeric columns remaining:")
print(non_numeric.columns)

In [ ]:
new.head()

In [ ]:
print(new["label_binary"].value_counts())

In [ ]:
normal = (new["label_binary"] == 0).sum()
covert = (new["label_binary"] == 1).sum()

print("Total Normal Samples :", normal)
print("Total Covert Samples :", covert)
print("Total Dataset Samples:", len(new))

In [ ]:
print(new["label_binary"].value_counts(normalize=True) * 100)

In [ ]:
import os

base_path = "/content/drive/MyDrive/Network_steganography_Project"
os.makedirs(base_path, exist_ok=True)

In [ ]:
new.to_csv(f"{base_path}/preprocessed_master_dataset.csv", index=False)
print("Saved full dataset")

In [ ]:
train_df.to_csv(f"{base_path}/train_dataset.csv", index=False)
test_df.to_csv(f"{base_path}/test_dataset.csv", index=False)

In [ ]:
print(os.listdir(base_path))

In [ ]:
import pickle

with open("protocol_encoder.pkl","wb") as f:
    pickle.dump(protocol_encoder,f)

print("Protocol encoder saved")